In [1]:
import os
import re
import json
import time
import glob
import warnings
warnings.filterwarnings("ignore")

import subprocess
import sys
import rasterio
import geopandas
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns

from rasterio.enums import Resampling
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (accuracy_score, classification_report,confusion_matrix, r2_score, mean_squared_error)
from scipy import stats as scipy_stats
from datetime import datetime
from pathlib import Path

In [2]:
plt.rcParams.update({
    "font.family":      "DejaVu Sans",
    "font.size":        11,
    "axes.titlesize":   13,
    "figure.dpi":       120,
    "figure.facecolor": "white",
})

In [3]:
rawDataDir = "data/raw"
outDataDir = "out"

for d in ["stacked","ndvi","ndwi","ndbi","classmaps","change","graphs","stats","models"]:
    os.makedirs(os.path.join(outDataDir, d), exist_ok=True)


In [4]:
scaleFactor = 0.0000275
offset = -0.2

bandNames = {"B1":"Coastal","B2":"Blue","B3":"Green","B4":"Red","B5":"NIR","B6":"SWIR1","B7":"SWIR2"}
classNames = {0:"No Change", 1:"Deforestation", 2:"Reforestation"}
classColors = {-1:"lightgray", 0:"blue", 1:"green",
                2:"yellow",  3:"red"}

ndviThreshold = 0.25
qaCloudBit = 0x0008
qaShadowBit = 0x0010
qaCirrusBit = 0x0004
qaFillBit = 0x0001
qaDilatedCloudBit = 0x0002